# 📸 Instagram AI Agent
**Full-featured Instagram automation that behaves like a human.**

### Features:
- 🔐 Login (session saved, no repeated logins)
- 📤 Upload Photos, Videos & Reels
- ❤️ Like & Unlike Posts
- 💬 Comment, Reply to Comments
- 👥 Follow / Unfollow Users
- 📊 View Post Analytics (likes, views, comments)
- 👁️ Watch Stories
- 📩 Send Direct Messages
- 🔍 Search by Hashtag
- ⏰ Post Scheduler

> ⚠️ All actions use randomized human-like delays to avoid detection.

In [1]:
# ============================================================
# CELL 1: Imports & Human Behavior Engine
# ============================================================
import time
import random
import json
import os
import sys
from pathlib import Path
from datetime import datetime
from getpass import getpass

from instagrapi import Client
from instagrapi.exceptions import LoginRequired, PleaseWaitFewMinutes
from instagrapi.types import Location

import colorama
from colorama import Fore, Style
colorama.init()

# ── Human-like delay engine ──────────────────────────────────
def human_delay(min_sec: float = 1.5, max_sec: float = 4.5):
    """Random sleep that mimics human reaction time."""
    t = random.uniform(min_sec, max_sec)
    print(f"{Fore.CYAN}  ⏳ waiting {t:.1f}s...{Style.RESET_ALL}")
    time.sleep(t)

def long_delay(min_sec: float = 8.0, max_sec: float = 20.0):
    """Longer pause between heavy actions."""
    t = random.uniform(min_sec, max_sec)
    print(f"{Fore.YELLOW}  😴 long pause {t:.1f}s...{Style.RESET_ALL}")
    time.sleep(t)

def burst_delay():
    """Micro-burst delay (scrolling simulation)."""
    time.sleep(random.uniform(0.3, 0.9))

def log(msg: str, level: str = "info"):
    colors = {"info": Fore.GREEN, "warn": Fore.YELLOW, "error": Fore.RED, "data": Fore.MAGENTA}
    c = colors.get(level, Fore.WHITE)
    print(f"{c}[{datetime.now().strftime('%H:%M:%S')}] {msg}{Style.RESET_ALL}")

log("✅ Imports & human engine loaded.")

[23:01:31] ✅ Imports & human engine loaded.


In [2]:
# ============================================================
# CELL 2: Configuration & Login
# ============================================================
USERNAME   = input("Instagram username: ")
PASSWORD   = getpass("Instagram password: ")
SESSION_FILE = Path(f"session_{USERNAME}.json")

cl = Client()
cl.delay_range = [2, 5]   # instagrapi built-in delay between requests

# Randomize device fingerprint
cl.set_locale("en_US")
cl.set_timezone_offset(-5 * 3600)   # EST – change to match your timezone

def instagram_login(cl: Client, username: str, password: str, session_file: Path) -> bool:
    if session_file.exists():
        log("Loading saved session...")
        try:
            cl.load_settings(session_file)
            cl.login(username, password)
            cl.get_timeline_feed()     # probe to validate session
            log("✅ Session restored, no re-login needed.")
            return True
        except LoginRequired:
            log("Session expired, logging in fresh...", "warn")
            session_file.unlink(missing_ok=True)

    log("Logging in fresh...")
    human_delay(2, 4)
    cl.login(username, password)
    cl.dump_settings(session_file)
    log("✅ Logged in & session saved.")
    return True

instagram_login(cl, USERNAME, PASSWORD, SESSION_FILE)
log(f"Logged in as @{USERNAME}", "data")

[23:07:05] Logging in fresh...
  ⏳ waiting 3.6s...


UnknownError: Invalid Parameters

In [ ]:
# ============================================================
# CELL 3: Upload Photo
# ============================================================
def upload_photo(image_path: str, caption: str = "", location_name: str = None):
    """
    Upload a photo to Instagram.
    image_path : local path to image file
    caption    : post caption (use \\n for line breaks)
    """
    log(f"Uploading photo: {image_path}")
    human_delay(3, 7)

    extra = {}
    if location_name:
        results = cl.location_search(location_name)
        if results:
            extra["location"] = results[0]

    media = cl.photo_upload(image_path, caption=caption, **extra)
    log(f"✅ Photo uploaded! media_id={media.id}", "data")
    long_delay()
    return media

# ── Example (uncomment to use) ────────────────────────────────
# m = upload_photo("photo.jpg", caption="Good morning! 🌞\n\n#morning #vibes", location_name="New York")

In [ ]:
# ============================================================
# CELL 4: Upload Video / Reel
# ============================================================
def upload_video(video_path: str, caption: str = "", thumbnail_path: str = None, as_reel: bool = False):
    """
    Upload a video or reel.
    as_reel=True  → posts as Instagram Reel
    as_reel=False → regular video post
    """
    log(f"Uploading {'Reel' if as_reel else 'Video'}: {video_path}")
    human_delay(5, 12)

    if as_reel:
        media = cl.clip_upload(video_path, caption=caption)
    else:
        kwargs = {"caption": caption}
        if thumbnail_path:
            kwargs["thumbnail"] = thumbnail_path
        media = cl.video_upload(video_path, **kwargs)

    log(f"✅ Video uploaded! media_id={media.id}", "data")
    long_delay(15, 30)
    return media

# ── Example ──────────────────────────────────────────────────
# m = upload_video("reel.mp4", caption="New reel! 🔥 #fyp #reels", as_reel=True)

In [ ]:
# ============================================================
# CELL 5: Like & Unlike Posts
# ============================================================
def like_posts_by_hashtag(hashtag: str, count: int = 5):
    """Browse hashtag feed and like posts like a real user."""
    log(f"Browsing #{hashtag}, will like {count} posts...")
    human_delay(2, 5)

    medias = cl.hashtag_medias_recent(hashtag, amount=count * 3)
    liked = 0
    for media in medias:
        if liked >= count:
            break
        burst_delay()                          # simulate scrolling
        if random.random() < 0.75:             # 75% chance to like (human doesn't like everything)
            cl.media_like(media.id)
            log(f"  ❤️  Liked post {media.id} by @{media.user.username}")
            liked += 1
            human_delay(2, 6)
        else:
            log(f"  ⏭️  Skipped post (human skip)")
    log(f"✅ Liked {liked} posts.", "data")

def like_post(media_id: str):
    """Like a single post by media_id."""
    human_delay()
    cl.media_like(media_id)
    log(f"❤️  Liked {media_id}")

def unlike_post(media_id: str):
    human_delay()
    cl.media_unlike(media_id)
    log(f"💔 Unliked {media_id}")

# ── Example ──────────────────────────────────────────────────
# like_posts_by_hashtag("python", count=5)

In [ ]:
# ============================================================
# CELL 6: Comment & Reply to Comments
# ============================================================
def human_type_comment(comment: str) -> str:
    """Simulate typing delay, return comment (with occasional typo-fix simulation)."""
    time.sleep(len(comment) * random.uniform(0.04, 0.09))
    return comment

def comment_on_post(media_id: str, text: str):
    """Post a comment on a media."""
    log(f"Typing comment on {media_id}...")
    human_delay(2, 5)
    text = human_type_comment(text)
    comment = cl.media_comment(media_id, text)
    log(f"✅ Commented: '{text}' → comment_id={comment.pk}", "data")
    long_delay(5, 15)
    return comment

def reply_to_comment(media_id: str, comment_id: str, username: str, reply_text: str):
    """Reply to a specific comment."""
    log(f"Replying to comment {comment_id}...")
    human_delay(3, 6)
    full_reply = f"@{username} {reply_text}"
    full_reply = human_type_comment(full_reply)
    comment = cl.media_comment(media_id, full_reply, replied_to_comment_id=int(comment_id))
    log(f"✅ Replied: '{full_reply}'", "data")
    long_delay(5, 12)
    return comment

def get_all_comments(media_id: str, amount: int = 20):
    """Fetch and display comments on a post."""
    human_delay(1, 3)
    comments = cl.media_comments(media_id, amount=amount)
    log(f"📝 {len(comments)} comments on {media_id}:", "data")
    for c in comments:
        print(f"  [{c.pk}] @{c.user.username}: {c.text}")
    return comments

def reply_to_all_comments(media_id: str, reply_generator):
    """
    reply_generator: a callable that takes (username, comment_text) → reply string
    Example:
        reply_to_all_comments(media_id, lambda u, t: f"Thanks @{u}! 🙏")
    """
    comments = get_all_comments(media_id)
    for c in comments:
        reply_text = reply_generator(c.user.username, c.text)
        if reply_text:
            reply_to_comment(media_id, str(c.pk), c.user.username, reply_text)
            human_delay(8, 20)

# ── Example ──────────────────────────────────────────────────
# comment_on_post("MEDIA_ID_HERE", "Amazing shot! 🔥")
# reply_to_all_comments("MEDIA_ID_HERE", lambda u, t: "Thank you so much! 😊")

In [ ]:
# ============================================================
# CELL 7: Follow & Unfollow
# ============================================================
def follow_user(username: str):
    human_delay(2, 5)
    user_id = cl.user_id_from_username(username)
    result  = cl.user_follow(user_id)
    log(f"➕ Followed @{username} → {result}")
    long_delay(5, 15)

def unfollow_user(username: str):
    human_delay(2, 5)
    user_id = cl.user_id_from_username(username)
    result  = cl.user_unfollow(user_id)
    log(f"➖ Unfollowed @{username} → {result}")
    long_delay(5, 15)

def follow_hashtag_users(hashtag: str, count: int = 3):
    """Follow users who recently posted with a hashtag."""
    log(f"Finding users from #{hashtag} to follow...")
    medias = cl.hashtag_medias_recent(hashtag, amount=count * 2)
    followed = 0
    for m in medias:
        if followed >= count:
            break
        if random.random() > 0.5:    # don't follow every single one
            follow_user(m.user.username)
            followed += 1
    log(f"✅ Followed {followed} users.")

# ── Example ──────────────────────────────────────────────────
# follow_user("some_username")
# follow_hashtag_users("photography", count=3)

In [ ]:
# ============================================================
# CELL 8: View Stories
# ============================================================
def watch_stories_of_user(username: str):
    """Simulate watching all stories of a user."""
    log(f"Watching stories of @{username}...")
    human_delay(2, 4)
    user_id = cl.user_id_from_username(username)
    stories = cl.user_stories(user_id)
    if not stories:
        log(f"No active stories for @{username}", "warn")
        return
    seen_ids = [s.id for s in stories]
    cl.story_seen(seen_ids)
    log(f"✅ Watched {len(stories)} stories of @{username}", "data")
    human_delay(2, 5)

def watch_following_stories(count: int = 5):
    """Watch stories from people you follow."""
    feed = cl.reels_tray()   # stories tray
    watched = 0
    for tray in feed[:count]:
        try:
            cl.story_seen([s.id for s in tray.items])
            log(f"  👁️  Watched stories from @{tray.user.username}")
            watched += 1
            human_delay(3, 8)
        except Exception as e:
            log(f"  ⚠️  {e}", "warn")
    log(f"✅ Watched stories from {watched} accounts.", "data")

# ── Example ──────────────────────────────────────────────────
# watch_stories_of_user("natgeo")
# watch_following_stories(count=3)

In [ ]:
# ============================================================
# CELL 9: Send Direct Messages
# ============================================================
def send_dm(username: str, message: str):
    """Send a direct message."""
    log(f"Sending DM to @{username}...")
    human_delay(3, 7)
    human_type_comment(message)   # simulate typing time
    user_id  = cl.user_id_from_username(username)
    thread   = cl.direct_send(message, user_ids=[user_id])
    log(f"✅ DM sent to @{username}! thread_id={thread.id}", "data")
    long_delay(10, 25)
    return thread

def reply_to_dm_thread(thread_id: str, message: str):
    """Reply to an existing DM thread."""
    log(f"Replying to thread {thread_id}...")
    human_delay(2, 5)
    human_type_comment(message)
    cl.direct_send(message, thread_ids=[thread_id])
    log(f"✅ Replied to thread.", "data")
    long_delay()

def get_inbox(amount: int = 10):
    """Fetch DM inbox."""
    human_delay()
    threads = cl.direct_threads(amount=amount)
    log(f"📩 {len(threads)} DM threads:", "data")
    for t in threads:
        users = ", ".join([u.username for u in t.users])
        print(f"  [{t.id}] With: {users}")
    return threads

# ── Example ──────────────────────────────────────────────────
# send_dm("some_user", "Hey! Loved your content 🔥")
# get_inbox()

In [ ]:
# ============================================================
# CELL 10: Post Analytics
# ============================================================
def get_post_analytics(media_id: str):
    """Show full analytics for a post."""
    human_delay(1, 3)
    media = cl.media_info(media_id)
    print("\n" + "═" * 45)
    print(f"  📊 Analytics for Post {media_id}")
    print("═" * 45)
    print(f"  👤 Author        : @{media.user.username}")
    print(f"  ❤️  Likes         : {media.like_count}")
    print(f"  💬 Comments      : {media.comment_count}")
    if hasattr(media, 'play_count') and media.play_count:
        print(f"  ▶️  Video Views   : {media.play_count}")
    print(f"  📅 Posted        : {media.taken_at}")
    print(f"  📝 Caption       : {str(media.caption_text)[:80]}...")
    print("═" * 45 + "\n")
    return media

def get_my_posts_analytics(count: int = 10):
    """Analytics for your own recent posts."""
    log("Fetching your recent posts...")
    user_id = cl.user_id_from_username(USERNAME)
    medias  = cl.user_medias(user_id, amount=count)
    for m in medias:
        get_post_analytics(str(m.id))
        burst_delay()

# ── Example ──────────────────────────────────────────────────
# get_my_posts_analytics(count=5)
# get_post_analytics("MEDIA_ID_HERE")

In [ ]:
# ============================================================
# CELL 11: Search & Explore by Hashtag
# ============================================================
def explore_hashtag(hashtag: str, count: int = 10):
    """Browse hashtag and display posts."""
    log(f"Exploring #{hashtag}...")
    human_delay(1, 3)
    medias = cl.hashtag_medias_top(hashtag, amount=count)
    print(f"\n🔍 Top posts for #{hashtag}:")
    for m in medias:
        print(f"  • [{m.id}] @{m.user.username} — ❤️ {m.like_count} 💬 {m.comment_count}")
        burst_delay()
    return medias

def search_users(query: str, count: int = 5):
    """Search Instagram users by name."""
    human_delay(1, 3)
    users = cl.search_users(query)[:count]
    print(f"\n👥 Users matching '{query}':")
    for u in users:
        print(f"  • @{u.username} — {u.full_name} ({u.follower_count} followers)")
    return users

# ── Example ──────────────────────────────────────────────────
# explore_hashtag("travel", count=8)
# search_users("National Geographic")

In [ ]:
# ============================================================
# CELL 12: Post Scheduler
# ============================================================
import schedule as sched
import threading

scheduled_jobs = []

def schedule_photo_post(image_path: str, caption: str, hour: int, minute: int):
    """Schedule a photo upload at a specific time (24h format)."""
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Scheduled post firing at {time_str}")
        upload_photo(image_path, caption)
    sched.every().day.at(time_str).do(job)
    log(f"📅 Photo scheduled for {time_str} daily.")

def schedule_video_post(video_path: str, caption: str, hour: int, minute: int, as_reel: bool = True):
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Scheduled video firing at {time_str}")
        upload_video(video_path, caption, as_reel=as_reel)
    sched.every().day.at(time_str).do(job)
    log(f"📅 Video scheduled for {time_str} daily.")

def run_scheduler():
    """Run the scheduler in a background thread."""
    def _loop():
        log("🕐 Scheduler started in background thread.")
        while True:
            sched.run_pending()
            time.sleep(30)
    t = threading.Thread(target=_loop, daemon=True)
    t.start()
    log("✅ Scheduler running. Keep notebook open.")

# ── Example ──────────────────────────────────────────────────
# schedule_photo_post("morning.jpg", "Good morning! ☀️", hour=9, minute=0)
# run_scheduler()

In [ ]:
# ============================================================
# CELL 13: Full Daily Routine (One-Click Run)
# ============================================================
def daily_routine(hashtags: list, like_per_tag: int = 3, watch_stories: int = 3):
    """
    Simulate a human daily Instagram session:
     1. Watch stories
     2. Like posts from hashtags
     3. Reply to own comments
    """
    log("🌅 Starting daily Instagram routine...")

    # 1. Watch stories
    log("Step 1: Watching stories...")
    try:
        watch_following_stories(count=watch_stories)
    except Exception as e:
        log(f"Stories error: {e}", "warn")

    long_delay(10, 20)

    # 2. Like posts from each hashtag
    log("Step 2: Liking hashtag posts...")
    for tag in hashtags:
        try:
            like_posts_by_hashtag(tag, count=like_per_tag)
        except Exception as e:
            log(f"#{tag} error: {e}", "warn")
        long_delay(15, 35)

    log("✅ Daily routine complete!", "data")

# ── Run it ───────────────────────────────────────────────────
# daily_routine(
#     hashtags=["photography", "travel", "food"],
#     like_per_tag=3,
#     watch_stories=4
# )